In [ ]:
__trill_node__ = {
    "id": "e1a123c1-8837-47a5-9b63-038e5ebcb530",
    "type": "DATA_LOADING",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    import pandas as pd

    df = pd.read_csv("/Users/jaideepnutalapati/Documents/GitHub/curio/docs/examples/data/Speed_Camera_Violations.csv")
    df.dropna(inplace=True)
    return df

_curio_output = _curio_node()

try:
    data_e1a123c1_8837_47a5_9b63_038e5ebcb530 = _curio_output
except NameError:
    data_e1a123c1_8837_47a5_9b63_038e5ebcb530 = None


In [ ]:
__trill_node__ = {
    "id": "cb340f83-0a4d-457a-be66-691672f330d3",
    "type": "COMPUTATION_ANALYSIS",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():

    input_0 = data_e1a123c1_8837_47a5_9b63_038e5ebcb530
    arg = input_0

    import pandas as pd

    df = arg

    df['VIOLATION DATE'] = pd.to_datetime(df['VIOLATION DATE'], format='%m/%d/%Y')

    df['Year'] = df['VIOLATION DATE'].dt.year

    yr_sum = (df.groupby(['CAMERA ID', 'Year'])['VIOLATIONS']
              .sum()
              .reset_index()
              .rename(columns={'VIOLATIONS': 'avg_violations'}))

    top_ids = (df.groupby('CAMERA ID')['VIOLATIONS']
                 .sum()
                 .sort_values(ascending=False)
                 .head(5)
                 .index
                 .tolist())

    yr_sum = yr_sum[yr_sum['CAMERA ID'].isin(top_ids)]

    camera_pos = (df.groupby('CAMERA ID')[['LATITUDE', 'LONGITUDE']]
                    .mean()
                    .reset_index())

    yr_sum = yr_sum.merge(camera_pos, on='CAMERA ID')

    return yr_sum


_curio_output = _curio_node()

try:
    result_cb340f83_0a4d_457a_be66_691672f330d3 = _curio_output
except NameError:
    result_cb340f83_0a4d_457a_be66_691672f330d3 = None


In [ ]:
__trill_node__ = {
    "id": "c2ba6e0e-e239-4167-a382-ffd1993cb3da",
    "type": "VIS_VEGA",
    "in": "DEFAULT",
    "out": "DEFAULT"
}

def _curio_node():


    input_data = result_cb340f83_0a4d_457a_be66_691672f330d3

    spec = {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      "data": { "name": "table" },
      "config": { "bar": { "continuousBandSize": 18 } },
      "hconcat": [
        {
          "width": 320,
          "height": 260,
          "selection": { "yrBrush": { "type": "interval", "encodings": ["x"] } },
          "mark": { "type": "bar" },
          "encoding": {
            "x": { "field": "Year", "type": "quantitative", "title": "Year" },
            "y": {
              "aggregate": "sum",
              "field": "avg_violations",
              "type": "quantitative",
              "title": "Total Violations"
            },
            "color": {
              "field": "CAMERA ID",
              "type": "nominal",
              "legend": { "title": "Camera ID" }
            }
          }
        },
        {
          "width": 320,
          "height": 260,
          "transform": [
            { "filter": { "selection": "yrBrush" } },
            {
              "aggregate": [
                { "op": "sum", "field": "avg_violations", "as": "total" }
              ],
              "groupby": ["Year"]
            },
            { "sort": { "field": "Year" } }
          ],
          "mark": { "type": "line", "point": True },
          "encoding": {
            "x": {
              "field": "Year",
              "type": "quantitative",
              "title": "Year (brush range)"
            },
            "y": {
              "field": "total",
              "type": "quantitative",
              "title": "Total Violations"
            }
          }
        }
      ]
    }

    values = input_data
    if hasattr(input_data, "to_dict"):
        values = input_data.to_dict(orient="records")

    if isinstance(spec, dict):
        spec["data"] = {"values": values}

    from IPython.display import display
    display({"application/vnd.vegalite.v5+json": spec, "text/plain": spec}, raw=True)

    return input_data


_curio_output = _curio_node()

try:
    result_c2ba6e0e_e239_4167_a382_ffd1993cb3da = _curio_output
except NameError:
    result_c2ba6e0e_e239_4167_a382_ffd1993cb3da = None
